In [58]:
# %% [1] — Imports
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import importlib

import solenoid_lib
importlib.reload(solenoid_lib)

from solenoid_lib import (
    solenoid_length,
    theta_solenoid,
    w_tape_mm,
    t_tape_mm,
    Je_tape,               
    solenoid_field_center,
    magnetic_energy,
    hoop_stress,
    je_max_stress_limited,
)

# ─────────────────────────────────────────────────────────────────────────────
# Parameters and Setup
# ─────────────────────────────────────────────────────────────────────────────
ri = 1.5                 # [m] inner radius
rf = 1.6           # [m] outer radius
solenoid_length = 4    # [m] length

sigma_limit_pa = 750e6 # [Pa] 750 MPa stress limit
# Note: Safety margin is now controlled entirely within solenoid_lib.py

max_iter = 100
tol_rel = 1e-4

# ─────────────────────────────────────────────────────────────────────────────
# 1. Pre-calculate the Mechanical Ceiling (independent of B or Cu)
# ─────────────────────────────────────────────────────────────────────────────
je_mech_si = je_max_stress_limited(ri, rf, solenoid_length, sigma_limit_pa)
je_mech_mm2 = je_mech_si / 1e6

# Start our guess strictly at the mechanical limit
je_mm2 = je_mech_mm2

# ─────────────────────────────────────────────────────────────────────────────
# 2. Self-Consistent Solver for SC Fraction and Current Density
# ─────────────────────────────────────────────────────────────────────────────
converged = False
iterations = 0

for i in range(max_iter):
    je_si = je_mm2 * 1e6
    
    # Calculate resulting central field with the current J
    b0 = solenoid_field_center(ri, rf, je_si, solenoid_length)
    
    # Call Je_tape with 0.0 copper fraction to simulate a 100% SC tape.
    # We now capture both outputs. The second output (je_100sc_mm2) 
    # automatically includes the margin defined inside solenoid_lib.py!
    jc_100sc_mm2, je_100sc_mm2 = Je_tape(b0, theta_solenoid, cu_frac=0.0)
    
    # Calculate the minimum amount of REBCO needed to carry our operating Je
    f_sc = je_mm2 / je_100sc_mm2
    
    if f_sc <= 1.0:
        # The mechanical limit requires a physically possible amount of SC.
        # Lock in the fractions and break out of the loop.
        cu_fraction = 1.0 - f_sc
        converged = True
        iterations = i + 1
        break
    else:
        # The required SC area is > 100%, meaning the current is too high.
        # Decrease the operating current density to the 100% SC limit and repeat.
        je_new_mm2 = je_100sc_mm2
        
        rel_change = abs(je_new_mm2 - je_mm2) / (abs(je_mm2) + 1e-12)
        je_mm2 = je_new_mm2
        
        if rel_change < tol_rel:
            cu_fraction = 0.0 # Bounded at 0% copper / 100% superconductor
            converged = True
            iterations = i + 1
            break

# ─────────────────────────────────────────────────────────────────────────────
# 3. Final Quantities at the Converged Operating Point
# ─────────────────────────────────────────────────────────────────────────────
je_si = je_mm2 * 1e6
b0 = solenoid_field_center(ri, rf, je_si, solenoid_length)
sigma_pa, _ = hoop_stress(ri, rf, je_si, solenoid_length)
E_mag = magnetic_energy(ri, rf, je_si, solenoid_length)

# Calculate the current density strictly within the superconducting material
j_sc_mm2 = je_mm2 / f_sc 

# ─────────────────────────────────────────────────────────────────────────────
# 4. SC-Only Equivalent Tape Length Calculations
# ─────────────────────────────────────────────────────────────────────────────
w_tape_m = w_tape_mm * 1e-3
t_tape_m = t_tape_mm * 1e-3

a_total = np.pi * (rf**2 - ri**2)
a_sc = a_total * f_sc

th_sc = np.sqrt(ri**2 + a_sc / np.pi) - ri
rf_sc = ri + th_sc

n_turns_radial_sc = th_sc / t_tape_m
r_mean_sc = ri + th_sc / 2.0

pancake_length_sc = n_turns_radial_sc * 2 * np.pi * r_mean_sc
n_pancakes = solenoid_length / w_tape_m
total_length_sc = pancake_length_sc * n_pancakes

# ─────────────────────────────────────────────────────────────────────────────
# Print Results
# ─────────────────────────────────────────────────────────────────────────────
if f_sc <= 1.0 and iterations == 1:
    limit_status = f"Mechanical Limit ({sigma_limit_pa/1e6:.0f} MPa)"
else:
    limit_status = "Electromagnetic Limit (100% SC threshold reached)"

print(f"--- Solver Status ---")
print(f"  Converged              = {converged} (in {iterations} iterations)")
print(f"  Coil Limiting Factor   = {limit_status}\n")

print(f"--- Operating Coil Properties ---")
print(f"  Generated Field (B0)   = {b0:.2f} T")
print(f"  Hoop Stress            = {sigma_pa/1e6:.0f} MPa")
print(f"  Magnetic Energy        = {E_mag/1e6:.2f} MJ\n")

print(f"--- Current Densities ---")
print(f"  Coil Engineering Jo    = {je_mm2:.2f} A/mm²")
print(f"  Superconductor J_e_tape    = {j_sc_mm2:.2f} A/mm²\n")

print(f"--- Tape Configuration ---")
print(f"  Required SC Fraction   = {f_sc*100:.2f} %")
print(f"  Remaining Fraction  = {cu_fraction*100:.2f} %\n")

print(f"--- SC-Only Equivalent Length Requirements ---")
print(f"  SC-only Thickness      = {th_sc * 1e3:.2f} mm")
print(f"  Equivalent SC Turns/Pan= {n_turns_radial_sc:.0f}")
print(f"  Number of Pancakes     = {n_pancakes:.0f}")
print(f"  Single Pancake SC Len. = {pancake_length_sc:.2f} m")
print(f"  Total SC Conductor Len.= {total_length_sc / 1e3:.2f} km")


--- Solver Status ---
  Converged              = True (in 1 iterations)
  Coil Limiting Factor   = Mechanical Limit (750 MPa)

--- Operating Coil Properties ---
  Generated Field (B0)   = 11.21 T
  Hoop Stress            = 750 MPa
  Magnetic Energy        = 1496.35 MJ

--- Current Densities ---
  Coil Engineering Jo    = 112.86 A/mm²
  Superconductor J_e_tape    = 1502.47 A/mm²

--- Tape Configuration ---
  Required SC Fraction   = 7.51 %
  Remaining Fraction  = 92.49 %

--- SC-Only Equivalent Length Requirements ---
  SC-only Thickness      = 7.74 mm
  Equivalent SC Turns/Pan= 52
  Number of Pancakes     = 1000
  Single Pancake SC Len. = 487.70 m
  Total SC Conductor Len.= 487.70 km
